# Annotation — Шаг 3

Авторазметка данных из `data/clean/`.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from agents.annotation_agent import AnnotationAgent

clean_files = sorted(Path('../data/clean').glob('*.parquet'))
df = pd.read_parquet(clean_files[-1])
print(f'Загружено: {df.shape}')

In [ ]:
# Авторазметка
agent = AnnotationAgent(
    modality='text',
    confidence_threshold=0.85,
    task='classification',
    classes=['World', 'Sports', 'Business', 'Sci/Tech']
)
# Используем подвыборку для ноутбука
df_sample = df.sample(min(500, len(df)), random_state=42).reset_index(drop=True)
df_labeled = agent.auto_label(df_sample)
print(f'Размечено: {len(df_labeled)} строк')

In [ ]:
# Распределение confidence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_labeled['confidence'], bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(x=0.85, color='red', linestyle='--', label='threshold=0.85')
axes[0].set_title('Распределение уверенности')
axes[0].set_xlabel('Confidence')
axes[0].legend()

df_labeled['predicted_label'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Распределение predicted_label')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Топ-10 с наибольшей уверенностью
print('=== Топ-10 с наибольшей уверенностью ===')
display_cols = [c for c in ['text', 'predicted_label', 'confidence'] if c in df_labeled.columns]
top10 = df_labeled.nlargest(10, 'confidence')[display_cols]
top10['text'] = top10['text'].str[:80]
print(top10.to_string(index=False))

In [ ]:
# Топ-10 с наименьшей уверенностью (HITL кандидаты)
print('=== Топ-10 с наименьшей уверенностью (для HITL) ===')
bottom10 = df_labeled.nsmallest(10, 'confidence')[display_cols]
bottom10['text'] = bottom10['text'].str[:80]
print(bottom10.to_string(index=False))

In [ ]:
# HITL статистика
df_auto, df_review = agent.flag_for_review(df_labeled)

# Спецификация разметки
spec = agent.generate_spec(df_labeled, 'news classification')
print(spec[:500] + '...')

In [ ]:
# Экспорт в LabelStudio
paths = agent.export_to_labelstudio(df_labeled)
print('Экспортировано:', paths)

In [ ]:
# Метрики качества
metrics = agent.check_quality(df_labeled)
import json
print(json.dumps({k: v for k, v in metrics.items() if k != 'label_distribution'}, indent=2))

## Вывод

- Авторазметка завершена
- Примеры с confidence < 0.85 флагнуты для ручной проверки (HITL-1)
- Спецификация разметки: `reports/annotation_spec.md`
- Экспорт для LabelStudio: `exports/labelstudio_import.json`